In [13]:
import yfinance as yf
import pandas as pd

In [14]:
# Create a list of tickers
df = pd.read_csv("../../data/processed/stocks/stocks_transformed_verified.csv")
tickers = df["ticker"].tolist()
tickers

['000333.SZ',
 '002050.SZ',
 '002230.KQ',
 '002460.KS',
 '002594.SZ',
 '002979.SZ',
 '003021.SZ',
 '005380.KS',
 '005930.KS',
 '006400.KS',
 '018260.KS',
 '1810.T',
 '1COV.F',
 '2317.T',
 '2454.T',
 '2498.T',
 '300014.SZ',
 '300124.SZ',
 '300750.SZ',
 '3402.T',
 '373220.KS',
 '4188.T',
 '601100.SS',
 '601689.SS',
 '603501.SS',
 '603728.SS',
 '6268.T',
 '6324.T',
 '6481.T',
 '6506.T',
 '6594.T',
 '6752.T',
 '6762.T',
 '688017.SS',
 '688322.SS',
 '688698.SS',
 '688836.SS',
 '6954.T',
 '6981.T',
 '9880.T',
 'ABBN.SW',
 'ADI',
 'ALB',
 'ALGM',
 'ALNT',
 'AMBA',
 'AMD',
 'AME',
 'AMZN',
 'BAS.DE',
 'BMW.DE',
 'CEVA',
 'CGNX',
 'DD',
 'DHL.DE',
 'EVK.DE',
 'GOOGL',
 'GXO',
 'HSAI',
 'HXL',
 'IFX.DE',
 'INTC',
 'JBL',
 'LAZR',
 'LSCC',
 'LYC.F',
 'MBG.DE',
 'MELI',
 'META',
 'MMM',
 'MP',
 'MPWR',
 'MT',
 'NOVT',
 'NOW',
 'NVDA',
 'NVTS',
 'NXPI',
 'OUST',
 'PTC',
 'QCOM',
 'RBC',
 'ROK',
 'RRX',
 'SIE.DE',
 'SONY',
 'SQM',
 'STM',
 'TM',
 'TMV',
 'TRMB',
 'TSLA',
 'TXN',
 'U',
 'USAR',
 'UUU

In [15]:
# I create a list to store the KPI data for each ticker. I will create a DataFrame from this list at the end of the loop.
kpi_data = []

In [16]:
# Calculate KPIs for each stock.
for ticker in tickers:

    try:
        t = yf.Ticker(ticker)
        
        # KPI #1: PEG ratio.
        forward_EPS_growth = ((t.info['forwardEps'] - t.info['epsCurrentYear']) / t.info['epsCurrentYear']) * 100
        PEG_ratio = t.info['forwardPE'] / forward_EPS_growth

        # KPI #2: ROIC (ROIC = NOPAT / Invested Capital
        # 
        # --> NOPAT = EBIT * (1 - tax rate)
        # --> Invested Capital = Total Debt + Stockholders Equity - Cash and Cash Equivalents

        ebit = t.financials.loc['EBIT'].iloc[0]
        tax_Rate = t.financials.loc["Tax Rate For Calcs"].iloc[0]
        nopat = ebit * (1 - tax_Rate)

        tot_Debt = t.balance_sheet.loc["Total Debt"].iloc[0]
        tot_Equity = t.balance_sheet.loc["Stockholders Equity"].iloc[0]
        cash = t.balance_sheet.loc["Cash And Cash Equivalents"].iloc[0]
        invested_Capital = tot_Debt + tot_Equity - cash

        roic = (nopat / invested_Capital) * 100


        # I also get the company name
        name = t.info['shortName']


        # Add the KPIs to the list.
        kpi_data.append({
            "Ticker": ticker,
            "PEG Ratio (1Y)": round(PEG_ratio, 2),
            "ROIC (%)": round(roic, 2),
            "Name": name
        })

    except Exception as e:
        print(f"Could not get data for {ticker}: {e}")

Could not get data for 002230.KQ: 'forwardEps'
Could not get data for 002460.KS: 'forwardEps'
Could not get data for 002979.SZ: 'epsCurrentYear'
Could not get data for 005380.KS: 'forwardEps'
Could not get data for 005930.KS: 'forwardEps'
Could not get data for 006400.KS: 'forwardEps'
Could not get data for 018260.KS: 'forwardEps'
Could not get data for 1810.T: 'forwardEps'
Could not get data for 2317.T: 'epsCurrentYear'
Could not get data for 2454.T: 'forwardEps'
Could not get data for 2498.T: 'forwardEps'
Could not get data for 3402.T: 'epsCurrentYear'
Could not get data for 373220.KS: 'forwardEps'
Could not get data for 4188.T: 'epsCurrentYear'
Could not get data for 603728.SS: 'epsCurrentYear'
Could not get data for 6268.T: 'epsCurrentYear'
Could not get data for 6324.T: 'epsCurrentYear'
Could not get data for 6481.T: 'epsCurrentYear'
Could not get data for 6506.T: 'epsCurrentYear'
Could not get data for 6594.T: 'epsCurrentYear'
Could not get data for 6752.T: 'epsCurrentYear'
Could

In [17]:
df_kpis = pd.DataFrame(kpi_data)
df_kpis

,Ticker,PEG Ratio (1Y),ROIC (%),Name
0,000333.SZ,1.60,22.57,MIDEA GROUP
1,002050.SZ,1.84,20.21,SANHUA
2,002594.SZ,0.44,12.39,BYD
3,003021.SZ,0.95,7.45,ZHAOWEI
4,1COV.F,0.07,-3.00,Covestro AG I
...,...,...,...,...
64,UUUU,-0.08,-7.76,Energy Fuels Inc
65,VPG,0.20,2.22,"Vishay Precision Group, Inc."
66,WMT,2.76,15.63,Walmart Inc.
67,WOLF,0.26,NaN,"Wolfspeed, Inc."


## Classification

Je classe chaque ticker dans une catégorie pour regrouper des tickers aux caractéristiques similaires.

- Efficace & Sous-évaluée (ES) : PEG ≤ 1 + ROIC ≥ médiane
- Efficace & Valuée (EV) : PEG > 1 + ROIC ≥ médiane
- Peu efficace & Sous-évaluée (PS) : PEG ≤ 1 + ROIC < médiane
- Peu efficace & Valuée (PV) : PEG > 1 + ROIC < médiane

Inclassables : tickers avec un PEG > 10 ou < 0.

In [18]:
# Masks
peg_positif = df_kpis["PEG Ratio (1Y)"] > 0
peg_less_than_10 = df_kpis["PEG Ratio (1Y)"] <= 10
has_peg = df_kpis["PEG Ratio (1Y)"].notna()
has_roic = df_kpis["ROIC (%)"].notna()

# Filter the dataframe to remove outliers and tickers missing data.
df_ok = df_kpis[(peg_positif) & (peg_less_than_10) & (has_peg) & (has_roic)]

# Calculate the median ROIC.
roic_median = round(df_ok["ROIC (%)"].median(), 2)
roic_median

np.float64(7.46)

In [19]:
# Assign a category to each ticker.
def get_category(row):
    peg = row["PEG Ratio (1Y)"]
    roic = row["ROIC (%)"]

    if peg <= 1 and roic >= roic_median:
        return "ES"
    elif peg > 1 and roic >= roic_median:
        return "EV"
    elif peg <= 1 and roic < roic_median:
        return "PS"
    else:
        return "PV"


df_ok["Category"] = df_ok.apply(get_category, axis=1)
df_ok

,Ticker,PEG Ratio (1Y),ROIC (%),Name,Category
0,000333.SZ,1.60,22.57,MIDEA GROUP,EV
1,002050.SZ,1.84,20.21,SANHUA,EV
2,002594.SZ,0.44,12.39,BYD,ES
3,003021.SZ,0.95,7.45,ZHAOWEI,PS
4,1COV.F,0.07,-3.00,Covestro AG I,PS
5,300014.SZ,0.26,7.47,EVE,ES
6,300124.SZ,0.66,16.55,INOVANCE,ES
7,300750.SZ,0.52,64.20,CATL,ES
8,601100.SS,1.00,32.41,JIANGSU HENGLI HYDRAULIC CO LTD,ES
9,601689.SS,1.18,12.14,NINGBO TUOPU GROUP CO LTD,EV


## Export

In [20]:
# Export
df_ok.to_csv("../../data/processed/stocks/stocks_kpis.csv", index=False)